<a href="https://colab.research.google.com/github/nhunggnguyen04/NCKH/blob/main/Phobert_fine_tuning_sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build Aspect Based Sentiment analysis cho bộ dữ liệu ABSA Tiki book review

- Ở đây mình sẽ fine tune mô hình phobert theo dataset ABSA TIKI BOOK Review (https://www.kaggle.com/datasets/nguynvntnpht/tiki-cleaned-book-reviews/data)
- Sau đó mình sẽ gán nhãn cho các bộ dữ liệu tiki review mình đã crawl từ trước


- Về bộ dữ liệu ABSA mình lấy ở trên Kaggle nó sẽ chia thành các aspect khác nhau (cái này tớ có nói với Nhung về ý tưởng ở trên tv rồi ý) --> thì mấy tuần qua mình đi đọc với học qua về cách fine tune mô hình Phobert✌ Vì tớ đánh giá kết quả của mô hình "wonrax phobert" trước cậu đưa cho tớ chưa được tối ưu vì nhiều nhãn bị gán sai

- Nhóm đã đi lọc sạch lại bộ dữ liệu trên kaggle vì quá "bẩn", hiện tại lọc được 7000 dữ liệu có thể sử dụng được

In [ ]:
# Truy cập drive để lấy dữ liệu
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Tải các thư viện cần thiết
## !pip install transformers
## !pip install torchvision
## !pip install Dataset

## Import mấy cái này nếu cần

!pip install underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 92.3 MB/s eta 0:00:00


In [ ]:
# Import các thư viện cần thiết
import os
os.environ["WANDB_DISABLED"] = "true"
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.utils import resample
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.model_selection import train_test_split
from datasets import Dataset
from underthesea import word_tokenize
from datasets import Value
print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
# Load model sử dụng
MODEL_NAME = "vinai/phobert-base-v2"
TEXT_COL = "content"
# nhãn này sẽ thay đổi trong quá trình mình tune từng model 1 cho từng aspect
LABEL_COL = "as_service"   # đổi lần lượt thành các aspect khác ở trong file data nhen

In [ ]:
# Gán trước mấy cái model để tý xuống dưỡi mình chạy
# Khởi chạy model Phobert
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4    # Có 3 nhãn tương ứng Tích cực, Tiêu cực, Trung tính
)

# Khởi chạy tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# Load bộ dữ liệu training
df = pd.read_excel('/content/drive/MyDrive/Data_fine_tuning/train_clean_extensive_dup.xlsx')

# Fill các dữ liệu thiếu
if 'content' in df.columns:
    df['content'] = df['content'].fillna("")

display(df)

,content,sentiment_llm,as_content,as_physical,as_price,as_packaging,as_delivery,as_service
0,sách khó hiểu. học theo bác ấy. là học về thốn...,0.0,0,NaN,NaN,NaN,NaN,NaN
1,"Giấy siêu đẹp, đúng là tiền nào của nấy. Về nộ...",2.0,NaN,NaN,NaN,NaN,NaN,NaN
2,"sách tạm được, giao hàng nhanh",1.0,NaN,2.0,NaN,NaN,2.0,NaN
3,Phí giao hàng đắc hơn gía sản phẩm,0.0,NaN,NaN,NaN,NaN,0.0,NaN
4,cái tấm bưu ảnh (post card) bên trong truyện t...,0.0,NaN,0.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
15391,Nội dung sách khá tệ. Mình mua bộ này vì trước...,0.0,0,NaN,0.0,NaN,NaN,NaN
15392,Giao hàng quá lâu ! Thay đổi ngày giao 2 3 lần .,0.0,NaN,NaN,NaN,NaN,0.0,NaN
15393,"Sách gì đây ko biết, hy vọng Shop kiểm tra kỹ ...",0.0,NaN,NaN,NaN,NaN,NaN,NaN
15394,Sách màu sắc đẹp. Mình mua hai quyển được tặng...,1.0,NaN,2.0,0.0,NaN,NaN,NaN


In [ ]:
# Kiểm tra trực tiếp các giá trị trong cột nhãn
print("Số lượng giá trị NaN còn lại:", df[LABEL_COL].isna().sum())
print("Danh sách các giá trị duy nhất trong cột:", df[LABEL_COL].unique())
print("\nSố lượng cụ thể cho từng nhãn:")
display(df[LABEL_COL].value_counts())

# Hiển thị một vài dòng có nhãn là 3 để xác nhận
print("\nVí dụ các dòng được gán nhãn 3:")
display(df[df[LABEL_COL] == 3].head())

Số lượng giá trị NaN còn lại: 13952
Danh sách các giá trị duy nhất trong cột: [nan  0.  2.  1.  3.]

Số lượng cụ thể cho từng nhãn:


,count
as_service,
0.0,874
2.0,330
3.0,150
1.0,90



Ví dụ các dòng được gán nhãn 3:


,content,sentiment_llm,as_content,as_physical,as_price,as_packaging,as_delivery,as_service
4000,"Mình thấy sách đẹp lắm nhaa, tô bút lông lên t...",2.0,3,1.0,3.0,2.0,3.0,3.0
4001,"Sách nhỏ nhắn, bìa đẹp, dễ mang theo. Giao hàn...",1.0,0,2.0,3.0,3.0,2.0,3.0
4002,"Sách ngắn, đọc nhanh, khá ok. Ngắn gọn, nhiều ...",2.0,2,3.0,3.0,3.0,3.0,3.0
4004,"Giao hàng nhanh, nhưng mà sách vừa nhận về bị ...",0.0,3,0.0,3.0,3.0,2.0,3.0
4005,"gh rất nhanh, sách mới bao bọc kỹ lưỡng nhưng ...",1.0,3,0.0,3.0,2.0,2.0,3.0


> Trong dataset thì các aspect sẽ có 3 nhãn tích cực, tiêu cực, trung tính và NaN --> với nhãn Na thì có nghĩa là trong bình luận không đề cập đến aspect ấy

> Mình test các kiểu để cân bằng dữ liệu rồi, đã chuyển 1 nhãn riêng đã chuyển về trung tính nhưng kết quả thường bị overfit

In [ ]:
# Điền giá trị NaN bằng 3 và đảm bảo lưu lại thay đổi
df[LABEL_COL] = df[LABEL_COL].fillna(3)

# Thay thế các ký tự không hợp lệ khác
df[LABEL_COL] = df[LABEL_COL].replace(['?', 'nan', 'NaN', ''], 3)

# Ép kiểu sang integer
df[LABEL_COL] = df[LABEL_COL].astype(int)

print("Kiểm tra lại sau khi sửa:")
print("Giá trị duy nhất:", df[LABEL_COL].unique())
display(df[LABEL_COL].value_counts())

Kiểm tra lại sau khi sửa:
Giá trị duy nhất: [3 0 2 1]


,count
as_service,
3,14102
0,874
2,330
1,90


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
import pandas as pd

# Chia tập train/test/validation theo tỷ lệ 80/10/10
# Chia tập gốc (df) thành train (80%) và temp (20%)
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df[LABEL_COL],
    random_state=42
)

# Chia tiếp temp thành val (10%) và test (10%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df[LABEL_COL],
    random_state=42
)

# Thực hiện Oversampling trên tập train_df (Oversampling giúp cân bằng dữ liệu)
print(f"Oversampling cho tập train, cột: {LABEL_COL}")
max_count = train_df[LABEL_COL].value_counts().max()

resampled_dfs = []
for label in train_df[LABEL_COL].unique():
    df_label = train_df[train_df[LABEL_COL] == label]
    df_label_upsampled = resample(
        df_label,
        replace=True,
        n_samples=max_count,
        random_state=42
    )
    resampled_dfs.append(df_label_upsampled)

# Gộp lại và tráo dữ liệu
train_df = pd.concat(resampled_dfs).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Kích thước tập train (sau oversampling): {len(train_df)}")
print(f"Kích thước tập validation: {len(val_df)}")
print(f"Kích thước tập test: {len(test_df)}\n")

print("Phân bố nhãn trong tập Train:")
display(train_df[LABEL_COL].value_counts())

Oversampling cho tập train, cột: as_service
Kích thước tập train (sau oversampling): 45124
Kích thước tập validation: 1540
Kích thước tập test: 1540

Phân bố nhãn trong tập Train:


,count
as_service,
1,11281
2,11281
3,11281
0,11281


In [ ]:
# Tách từ có gạch chéo
def vietnamese_word_segmenter(text):
    if not isinstance(text, str) or text.strip() == '':
        return ''
    try:
        return word_tokenize(text, format='text')
    except Exception as e:
        return str(text)

# Áp dụng tách từ sử dụng segmenter cho Tiếng Việt
train_df['segmented_content'] = train_df['content'].apply(vietnamese_word_segmenter)
val_df['segmented_content'] = val_df['content'].apply(vietnamese_word_segmenter)
test_df['segmented_content'] = test_df['content'].apply(vietnamese_word_segmenter)

# Khởi tạo bộ dữ liệu Hugging Face
train_ds = Dataset.from_pandas(train_df[['content', LABEL_COL]], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[['content', LABEL_COL]], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[['content', LABEL_COL]], preserve_index=False)

print(train_df[['content', 'segmented_content']].head(1))

                                             content  \
0  Giao sách sai, trả lời vòng vo, đổ lỗi nhân vi...   

                                   segmented_content  
0  Giao sách sai , trả_lời vòng_vo , đổ lỗi nhân_...  


In [ ]:
# Biến các từ đã được segmented thành các token
def tokenize_function(examples):
    return tokenizer(
        examples['segmented_content'],
        padding='max_length',
        truncation=True,
        max_length=256
    )


# Keep only the necessary columns
train_subset = train_df[['segmented_content', LABEL_COL]]
val_subset = val_df[['segmented_content', LABEL_COL]]
test_subset = test_df[['segmented_content', LABEL_COL]]

# Convert to Hugging Face Dataset objects
train_dataset = Dataset.from_pandas(train_subset, preserve_index=False)
val_dataset = Dataset.from_pandas(val_subset, preserve_index=False)
test_dataset = Dataset.from_pandas(test_subset, preserve_index=False)

# Verify the train dataset
print("Train Dataset:")
print(train_dataset)
print(train_subset.head(1))




Train Dataset:
Dataset({
    features: ['segmented_content', 'as_service'],
    num_rows: 45124
})
                                   segmented_content  as_service
0  Giao sách sai , trả_lời vòng_vo , đổ lỗi nhân_...           1


In [ ]:
# Biến bộ dữ liệu thành token
def tokenize_function(examples):
    return tokenizer(
        examples['segmented_content'],
        padding='max_length',
        truncation=True,
        max_length=256
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

# Xoá cột text segmented đi để mô hình không nhầm lẫn
tokenized_train = tokenized_train.remove_columns(['segmented_content'])
tokenized_val = tokenized_val.remove_columns(['segmented_content'])
tokenized_test = tokenized_test.remove_columns(['segmented_content'])

# Verify
print("Tokenized Train Dataset Features:", tokenized_train.features)
print("\nFirst row of tokenized train dataset:")
print(tokenized_train[0])

Map:   0%|          | 0/45124 [00:00<?, ? examples/s]

Map:   0%|          | 0/1540 [00:00<?, ? examples/s]

Map:   0%|          | 0/1540 [00:00<?, ? examples/s]

Tokenized Train Dataset Features: {'as_service': Value('int64'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}

First row of tokenized train dataset:
{'as_service': 1, 'input_ids': [0, 6460, 713, 1069, 4, 799, 34836, 4, 888, 1210, 650, 8179, 1069, 5, 15621, 1361, 505, 94, 124, 78, 17, 108, 602, 5, 20704, 204, 4152, 381, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

> Oke đến bước này là có thể fine-tune mô hình từ bộ dữ liệu mình có được rồi

In [ ]:
# các parameter để train
training_args = TrainingArguments(
    output_dir="./sentiment_class",
    do_train=True,
    do_eval=True,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    greater_is_better=True,
    logging_dir="./train_logs",
    logging_steps=100,
    fp16=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

# F1_score, Accuracy
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

In [ ]:
import torch
from torch import nn
from transformers import Trainer, EarlyStoppingCallback
from datasets import Value
import numpy as np
from sklearn.utils import resample

# Rename and cast labels
if LABEL_COL in tokenized_train.column_names:
    tokenized_train = tokenized_train.rename_column(LABEL_COL, "labels")
if LABEL_COL in tokenized_val.column_names:
    tokenized_val = tokenized_val.rename_column(LABEL_COL, "labels")

tokenized_train = tokenized_train.cast_column("labels", Value("int64"))
tokenized_val = tokenized_val.cast_column("labels", Value("int64"))

# 3. Standard Trainer (No custom weighted loss)
model.config.problem_type = "single_label_classification"

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("\nRestarting training with oversampled data and standard loss...")
trainer.train()

Casting the dataset:   0%|          | 0/45124 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/1540 [00:00<?, ? examples/s]


Restarting training with oversampled data and standard loss...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.089412,0.232572,0.953896,0.876484
2,0.032610,0.218725,0.964286,0.876885
3,0.023287,0.240105,0.968182,0.876102
4,0.009110,0.242397,0.968831,0.897367
5,0.001820,0.247111,0.970779,0.901886


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=7055, training_loss=0.08457479051437723, metrics={'train_runtime': 2991.8204, 'train_samples_per_second': 75.412, 'train_steps_per_second': 2.358, 'total_flos': 2.968209115262976e+16, 'train_loss': 0.08457479051437723, 'epoch': 5.0})

In [ ]:
# Ensure the label column is named 'labels' for the trainer to recognize it as ground truth
if LABEL_COL in tokenized_test.column_names:
    tokenized_test_final = tokenized_test.rename_column(LABEL_COL, "labels")
else:
    tokenized_test_final = tokenized_test

# Cast the labels to int64 to avoid NotImplementedError for Float
tokenized_test_final = tokenized_test_final.cast_column("labels", Value("int64"))

# Predict using the renamed dataset
test_results = trainer.predict(tokenized_test_final)
pred_labels = np.argmax(test_results.predictions, axis=1)
true_labels = test_results.label_ids

# Check if true_labels exist before calculating metrics
if true_labels is not None:
    test_acc = accuracy_score(true_labels, pred_labels)
    print(f"Độ chính xác (accuracy) trên tập test {LABEL_COL}: {test_acc:.4f}")

    # Cập nhật danh sách nhãn để khớp với 4 lớp (0, 1, 2, 3)
    # 0: Tiêu cực, 1: Trung lập, 2: Tích cực, 3: Không đề cập/NaN
    class_names = ["Tiêu cực", "Trung lập", "Tích cực", "Không đề cập"]

    print(f"Báo cáo phân loại trên tập test {LABEL_COL}:")
    print(classification_report(true_labels, pred_labels, target_names=class_names))
else:
    print("Lỗi: Không tìm thấy nhãn thực tế (true labels) trong dataset.")

Casting the dataset:   0%|          | 0/1540 [00:00<?, ? examples/s]

Độ chính xác (accuracy) trên tập test as_service: 0.9760
Báo cáo phân loại trên tập test as_service:
              precision    recall  f1-score   support

    Tiêu cực       0.87      0.84      0.86        88
   Trung lập       1.00      0.89      0.94         9
    Tích cực       0.79      0.91      0.85        33
Không đề cập       0.99      0.99      0.99      1410

    accuracy                           0.98      1540
   macro avg       0.91      0.91      0.91      1540
weighted avg       0.98      0.98      0.98      1540



In [ ]:
# Lưu lại mô hình đã được fine-tune (tốt nhất) và tokenizer
trainer.save_model("./best_phobert_model")
tokenizer.save_pretrained("./best_phobert_model")

# In ra kết quả mô hình tốt nhất trên tập validation
best_acc = trainer.state.best_metric
print(f"Độ chính xác tốt nhất trên tập validation: {best_acc:.4f}")
print("Checkpoint mô hình tốt nhất được lưu tại:", trainer.state.best_model_checkpoint)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Độ chính xác tốt nhất trên tập validation: 0.9708
Checkpoint mô hình tốt nhất được lưu tại: ./sentiment_class/checkpoint-7055


#Gán nhãn bộ dữ liệu tiki gốc

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from underthesea import word_tokenize
from tqdm.auto import tqdm

# 1. Tải mô hình và tokenizer
model_dir = "./best_phobert_model"
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# 2. Đọc dữ liệu
df_unlabel = pd.read_csv('/content/drive/MyDrive/tiki_reviews_with_sentiment_final.csv')
# Xử lý giá trị rỗng
df_unlabel['content'] = df_unlabel['content'].fillna('')

# 3. Hàm tiền xử lý (tách từ)
def vietnamese_word_segmenter(text):
    if not isinstance(text, str) or text.strip() == '':
        return ''
    try:
        return word_tokenize(text, format='text')
    except Exception:
        return str(text)

# THỰC HIỆN TÁCH TỪ (Bắt buộc vì file CSV gốc chưa có cột này)
#print("Đang tách từ...")
#df_unlabel['segmented_content'] = df_unlabel['content'].apply(vietnamese_word_segmenter)

# 4. Tạo Pytorch Dataset để xử lý batch
class ReviewDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=256):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        inputs = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0)
        }

dataset = ReviewDataset(df_unlabel['segmented_content'].tolist(), tokenizer)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

# 5. Dự đoán
all_preds = []
all_probs = []

print("Đang dự đoán cảm xúc...")
with torch.no_grad():
    for batch in tqdm(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = F.softmax(logits, dim=1)

        preds = torch.argmax(logits, dim=1).cpu().numpy()
        batch_probs = probs.cpu().numpy()

        all_preds.extend(preds)
        all_probs.extend(batch_probs)

# 6. Gán nhãn và lưu kết quả
all_probs = np.array(all_probs)

label_id = LABEL_COL + "_label_id"
df_unlabel[label_id] = all_preds
df_unlabel[LABEL_COL + "_score_negative"] = all_probs[:, 0]
df_unlabel[LABEL_COL + "_score_neutral"] = all_probs[:, 1]
df_unlabel[LABEL_COL + "_score_positive"] = all_probs[:, 2]
df_unlabel[LABEL_COL + "_score_na"] = all_probs[:, 3]

save_path = '/content/drive/MyDrive/tiki_reviews_with_sentiment_final.csv'
df_unlabel.to_csv(save_path, index=False)
print(f"Đã lưu kết quả tại: {save_path}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Đang dự đoán cảm xúc...


  0%|          | 0/2406 [00:00<?, ?it/s]

Đã lưu kết quả tại: /content/drive/MyDrive/tiki_reviews_with_sentiment_final.csv


In [ ]:
df_unlabel[label_id].value_counts()

,count
as_service_label_id,
3,72824
2,3187
0,902
1,65
